# NeoMint Planner — Dataset Validation

Validates the structured JSONL training data for the NeoMint Planner model.

**Checks:**
- JSON parse validity
- Required fields: id, messages, tool_manifest, target, labels
- Target conforms to action-plan schema (kind: plan/clarification/refusal)
- Tool names exist in the manifest
- No forbidden policy fields in model output
- No duplicate IDs
- Category and tool distribution statistics

## 1. Upload Data

Upload these files from the repo:
- `fine-tuning/data/train.jsonl`
- `fine-tuning/data/validation.jsonl`
- `fine-tuning/data/test.jsonl`
- `fine-tuning/schemas/action-plan.schema.json`
- `fine-tuning/schemas/tool-manifest.json`

In [ ]:
from google.colab import files
import json
from collections import Counter
from pathlib import Path

# Upload files
uploaded = files.upload()

## 2. Load Tool Manifest

In [ ]:
with open('tool-manifest.json', 'r') as f:
    manifest = json.load(f)

MANIFEST_TOOLS = {t['name'] for t in manifest['tools']}
print(f'Loaded {len(MANIFEST_TOOLS)} tools: {sorted(MANIFEST_TOOLS)}')

## 3. Validation Functions

In [ ]:
REQUIRED_FIELDS = {'id', 'messages', 'tool_manifest', 'target', 'labels'}
VALID_KINDS = {'plan', 'clarification', 'refusal'}
FORBIDDEN_FIELDS = {'requires_confirmation', 'permission', 'risk', 'execute', 'risk_level'}

def validate_entry(entry, line_num):
    issues = []
    for f in REQUIRED_FIELDS:
        if f not in entry:
            issues.append(f'Line {line_num}: Missing field "{f}"')
    if issues:
        return issues
    
    # Validate messages
    msgs = entry['messages']
    if len(msgs) < 2:
        issues.append(f'Line {line_num}: Need at least 2 messages')
    elif msgs[0].get('role') != 'system' or msgs[-1].get('role') != 'user':
        issues.append(f'Line {line_num}: Messages must start with system and end with user')
    
    # Validate target
    target = entry['target']
    kind = target.get('kind')
    if kind not in VALID_KINDS:
        issues.append(f'Line {line_num}: Invalid kind "{kind}"')
    elif kind == 'plan':
        if 'user_facing_summary' not in target:
            issues.append(f'Line {line_num}: Plan missing user_facing_summary')
        for i, act in enumerate(target.get('actions', [])):
            if act.get('tool') not in MANIFEST_TOOLS:
                issues.append(f'Line {line_num}, action[{i}]: Unknown tool "{act.get("tool")}"')
            for ff in FORBIDDEN_FIELDS:
                if ff in act:
                    issues.append(f'Line {line_num}: Forbidden field "{ff}" in action')
        for ff in FORBIDDEN_FIELDS:
            if ff in target:
                issues.append(f'Line {line_num}: Forbidden field "{ff}" in target')
    elif kind == 'clarification':
        if 'question' not in target or 'reason' not in target:
            issues.append(f'Line {line_num}: Clarification missing question/reason')
    elif kind == 'refusal':
        if 'message' not in target:
            issues.append(f'Line {line_num}: Refusal missing message')
    
    return issues

def validate_file(path):
    all_issues = []
    ids = []
    kinds = Counter()
    categories = Counter()
    tools = Counter()
    
    with open(path, 'r') as f:
        for i, line in enumerate(f, 1):
            entry = json.loads(line.strip())
            issues = validate_entry(entry, i)
            all_issues.extend(issues)
            if not issues:
                ids.append(entry['id'])
                kinds[entry['target']['kind']] += 1
                categories[entry['labels']['category']] += 1
                if entry['target']['kind'] == 'plan':
                    for act in entry['target']['actions']:
                        tools[act['tool']] += 1
    
    # Check duplicate IDs
    for eid, count in Counter(ids).items():
        if count > 1:
            all_issues.append(f'Duplicate ID: {eid} (x{count})')
    
    return all_issues, len(ids), kinds, categories, tools

## 4. Validate All Datasets

In [ ]:
for dataset in ['train.jsonl', 'validation.jsonl', 'test.jsonl']:
    if not Path(dataset).exists():
        print(f'Skipping {dataset} (not uploaded)')
        continue
    
    issues, total, kinds, categories, tools = validate_file(dataset)
    
    print(f'\n{"=" * 50}')
    print(f'{dataset}: {total} entries')
    print(f'{"=" * 50}')
    print(f'  Kinds: {dict(kinds)}')
    print(f'  Categories: {dict(categories)}')
    print(f'  Tools: {dict(tools)}')
    if issues:
        print(f'  ISSUES ({len(issues)}):')
        for issue in issues:
            print(f'    [!] {issue}')
    else:
        print(f'  No issues!')

## 5. Distribution Visualization

In [ ]:
import matplotlib.pyplot as plt

# Re-validate train to get stats
_, total, kinds, categories, tools = validate_file('train.jsonl')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Response kinds
axes[0].bar(kinds.keys(), kinds.values(), color=['#4CAF50', '#FF9800', '#F44336'])
axes[0].set_title('Response Kinds')
axes[0].set_ylabel('Count')

# Categories
cats = dict(categories.most_common())
axes[1].barh(list(cats.keys()), list(cats.values()), color='#2196F3')
axes[1].set_title('Categories')
axes[1].invert_yaxis()

# Tool usage
tool_data = dict(tools.most_common())
axes[2].barh(list(tool_data.keys()), list(tool_data.values()), color='#9C27B0')
axes[2].set_title('Tool Usage')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

print(f'\nTotal training examples: {total}')
print(f'Plans: {kinds["plan"]} ({kinds["plan"]/total*100:.1f}%)')
print(f'Clarifications: {kinds["clarification"]} ({kinds["clarification"]/total*100:.1f}%)')
print(f'Refusals: {kinds["refusal"]} ({kinds["refusal"]/total*100:.1f}%)')